# **Практическое занятие. Разбиение данных и валидация моделей выявления кибератак**

**Время:** 80 минут. **Среда:** Jupyter Notebook или Google Colab.

Основной код и вспомогательные функции предоставлены. Заполните ячейки и текстовые выводы. Каждая сравнительная схема заново использует development; случайный, групповой и временной варианты calibration — независимые учебные эксперименты, а не этапы одной модели.

## Исходные данные

Набор моделирует телеметрию офисных устройств и серверов за 120 дней с 1 сентября по 29 декабря 2025 года. 3600 событий образуют 480 групп разного размера (4, 6, 8 или 12 записей). После 60-го дня возрастает нормальная нагрузка. Атаки происходят неравномерно.

| Поля | Назначение |
|---|---|
| event_id | уникальный идентификатор строки |
| timestamp, window_start, window_end | регистрация и границы пятиминутного окна |
| host_id, host_role, user_id | узел, его роль и учётная запись |
| session_id | сессия связанных событий |
| incident_id, campaign_id | инцидент и кампания; пусты у нормальных событий |
| validation_group | сессия для нормы, целый инцидент для атаки |
| traffic_type | контекст активности |
| connections_per_min | соединения в минуту |
| unique_destinations | число уникальных адресатов |
| outbound_mb | исходящий трафик, Мбайт |
| failed_logins | число неуспешных входов |
| duration_sec | длительность активности, секунды |
| scenario, is_attack | учебный сценарий и метка 0/1 |
| dataset_part | development — первые 100 дней, final_test — последние 20 |
| day_index, hour, day_of_week | номер дня от 0, час, день недели от 0 |

Один атакующий инцидент включает две сессии; кампания объединяет несколько инцидентов. Поэтому группировка по session_id может разделить инцидент, а группировка по validation_group может разделить кампанию. Идентификаторы, время и контрольные поля не являются поведенческими признаками.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, StratifiedShuffleSplit,
    KFold, StratifiedKFold, RepeatedStratifiedKFold, GroupKFold,
    StratifiedGroupKFold, TimeSeriesSplit)
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
FEATURES = ['connections_per_min', 'unique_destinations', 'outbound_mb',
            'failed_logins', 'duration_sec']
SEED = 42

In [ ]:
df = pd.read_csv(data_path, parse_dates=['timestamp', 'window_start', 'window_end'])
df.head()

Вспомогательные функции.

In [ ]:
def overlap(first, second, column='validation_group'):
    # Пустые incident_id/campaign_id обозначают отсутствие инцидента/кампании.
    return len(set(first[column].dropna()) & set(second[column].dropna()))

def describe(name, data):
    return {'part': name, 'events': len(data), 'groups': data.validation_group.nunique(),
            'attacks': int(data.is_attack.sum()), 'attack_percent': 100 * data.is_attack.mean(),
            'start': data.timestamp.min(), 'end': data.timestamp.max()}

def partition_report(parts):
    print(pd.DataFrame([describe(name, part) for name, part in parts.items()]).to_string(index=False))
    names = list(parts)
    pairs = []
    for i, first in enumerate(names):
        for second in names[i + 1:]:
            pairs.append({'pair': first + ' / ' + second,
                'shared_events': overlap(parts[first], parts[second], 'event_id'),
                'shared_groups': overlap(parts[first], parts[second]),
                'shared_incidents': overlap(parts[first], parts[second], 'incident_id')})
    print(pd.DataFrame(pairs).to_string(index=False))

def analyze(data, splits):
    # split() возвращает позиции, поэтому используем iloc, а не loc.
    rows = []
    for fold, (tr, va) in enumerate(splits, 1):
        a, b = data.iloc[tr], data.iloc[va]
        rows.append({'fold': fold, 'train_size': len(a), 'validation_size': len(b),
            'validation_attacks': int(b.is_attack.sum()),
            'validation_attack_percent': b.is_attack.mean() * 100,
            'shared_groups': overlap(a, b), 'shared_incidents': overlap(a, b, 'incident_id'),
            'shared_campaigns': overlap(a, b, 'campaign_id'),
            'train_start': a.timestamp.min(), 'train_end': a.timestamp.max(),
            'validation_start': b.timestamp.min(), 'validation_end': b.timestamp.max()})
    return pd.DataFrame(rows)

def coverage(n, splits):
    counts = np.zeros(n, dtype=int)
    for _, va in splits:
        counts[va] += 1
    return counts

def show_blocks(name, data, splits):
    table = analyze(data, splits)
    print('\n' + name)
    print(table.drop(columns=['train_start', 'train_end', 'validation_start', 'validation_end']).round(3).to_string(index=False))
    counts = coverage(len(data), splits)
    print('Попадания в validation: min=', counts.min(), 'max=', counts.max(),
          'ни разу=', int((counts == 0).sum()))
    return table


## Задание 1. Аудит данных и отделение финального теста — 10 минут, 3 балла

1. Загрузить CSV с преобразованием временных столбцов; проверить размер, уникальность event_id и пропуски
2. Отделить development и final_test; показать числа событий, групп, атак и временные границы
3. Проверить отсутствие общих event_id, validation_group и incident_id между development и test
4. Указать пять поведенческих признаков и назначение остальных полей

**Проверяемый результат:** Таблица состава данных, 3000 событий development и 600 событий test, нулевые пересечения и объяснение назначения полей.


In [ ]:
# выведите размер таблицы, дубликаты, пропуски и уникальность event_id.
# сформируйте development и test по dataset_part с reset_index(drop=True).
# вызовите partition_report и проверьте порядок времени и пересечения.

**Ваш вывод:** Назовите допустимые пустые поля; объясните различие строк, признаков, меток и групп.


## Задание 2. Случайное разбиение и стратификация — 15 минут, 4 балла

1. Сформировать train, validation и calibration в долях 60/20/20 от development без стратификации и со стратификацией по is_attack
2. Сравнить доли атак, пересечения групп и инцидентов; найти конкретный разделённый инцидент
3. По готовому примеру StratifiedShuffleSplit проверить покрытие событий за пять повторений после фиксации calibration
4. По готовому примеру Bootstrap определить размер train, число уникальных событий и размер Out-of-Bag

**Проверяемый результат:** Два варианта 1800/600/600, отчёт об утечке связанных событий, счётчики покрытия и пояснение Bootstrap.


In [ ]:
# выполните два вызова train_test_split: test_size=0.20, затем 0.25.
# повторите со stratify=is_attack; сохраните random_parts для двух вариантов.
# вызовите partition_report; найдите общий incident_id в train и validation.
# Для следующей ячейки:
# parts = random_parts['Stratified Hold-out']

In [ ]:
# Готовый пример: calibration фиксирована на всех пяти повторениях.
remaining = pd.concat([parts['train'], parts['validation']]).reset_index(drop=True)
sss = StratifiedShuffleSplit(n_splits=5, test_size=0.25, random_state=SEED)
repeat_splits = list(sss.split(remaining[FEATURES], remaining.is_attack))
repeat_table = show_blocks('StratifiedShuffleSplit', remaining, repeat_splits)
repeat_counts = coverage(len(remaining), repeat_splits)

# Bootstrap: размер train равен размеру remaining, но строки выбираются с возвращением.
boot_train = resample(remaining, replace=True, n_samples=len(remaining), random_state=SEED)
oob = remaining.loc[~remaining.index.isin(boot_train.index)]
print('Bootstrap: строк=', len(boot_train), 'уникальных=', boot_train.index.nunique(),
      'Out-of-Bag=', len(oob), 'OOB %=', round(100 * len(oob) / len(remaining), 2))
print('Общих исходных event_id:', overlap(boot_train, oob, 'event_id'),
      'общих групп:', overlap(boot_train, oob))

**Ваш вывод:** Укажите реальные пересечения и число событий, не попавших в validation; объясните риск Bootstrap для связанных событий.


## Задание 3. Кросс-валидация по блокам и группам — 20 минут, 6 балла

1. Случайно выбрать 20 % уникальных validation_group для calibration целиком и оставить остальные группы в cv_pool
2. На одном cv_pool сравнить KFold, StratifiedKFold, GroupKFold и StratifiedGroupKFold с пятью блоками
3. Для каждого блока показать размеры, долю атак, общие группы, инциденты и кампании
4. Проверить однократное покрытие событий в KFold и GroupKFold; по готовому примеру RepeatedStratifiedKFold подтвердить покрытие дважды
5. На первом разбиении GroupKFold обучить StandardScaler только по train_idx и преобразовать validation_idx
6. Объяснить, почему разделение по инциденту не гарантирует разделение по кампании или по времени

**Проверяемый результат:** Четыре таблицы блоков, нулевые пересечения групп и инцидентов в групповых методах, проверка покрытия и корректное масштабирование.


In [ ]:
# выберите 20 % уникальных validation_group через sample(frac=0.20, random_state=42).
# целиком выделите group_calibration и cv_pool по маске isin.
# проверьте пересечения и выведите размеры и число групп.

In [ ]:
# создайте KFold и StratifiedKFold (5 блоков, shuffle=True, random_state=42).
# создайте GroupKFold(5) и StratifiedGroupKFold(5, shuffle=True, random_state=42).
# получите списки индексов all_splits[name] и таблицы block_tables[name].
# Используйте show_blocks, analyze и coverage из готовой ячейки.
# сравните shared_groups, shared_incidents и shared_campaigns; проверьте coverage==1.

In [ ]:
# Готовый пример повторной блочной проверки: каждый объект проверяется дважды.
repeated = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
repeated_splits = list(repeated.split(X, y))
repeated_counts = coverage(len(cv_pool), repeated_splits)
print('RepeatedStratifiedKFold: min=', repeated_counts.min(), 'max=', repeated_counts.max())
assert np.all(repeated_counts == 2)

In [ ]:
# возьмите первый блок all_splits['GroupKFold'].
# вызовите StandardScaler.fit_transform только для train_idx.
# вызовите transform для validation_idx; выведите scaler.mean_ и размеры.

**Ваш вывод:** Сопоставьте доли атак и пересечения четырёх методов. Как изменится проверка, если группой выбрать host_id или campaign_id?


## Задание 4. Хронологическая и временная кросс-валидация — 20 минут, 5 балла

1. Отсортировать development по времени; последние 20 % событий отложить для calibration
2. В оставшихся 2400 событиях сформировать хронологический Hold-out 80/20
3. Сравнить TimeSeriesSplit с расширяющимся и скользящим train: пять validation по 300 событий, max_train_size=800 для скользящего окна
4. Применить готовый генератор четырёх непересекающихся циклов: 12 дней train и 8 дней validation
5. Проверить хронологический порядок, пересечения групп и инцидентов и покрытие событий; построить временную схему окон средствами matplotlib
6. Объяснить неполное покрытие самых ранних событий и отсутствие гарантии сохранения долей атак

**Проверяемый результат:** Таблицы временных границ, график окон, 900 начальных событий вне validation в TimeSeriesSplit и вывод о групповых ограничениях.


In [ ]:
# отсортируйте development по timestamp.
# последние 20% отложите в time_calibration, остальные — в time_pool.
# последние 20% time_pool выделите в chron_validation, остальные — chron_train.
# вызовите partition_report для четырёх частей и проверьте временной порядок.

In [ ]:
def blocked_time_splits(data, n_splits=4, train_days=12, validation_days=8):
    # Каждый цикл занимает 20 календарных дней. Интервалы полуоткрытые.
    origin = data.timestamp.min().normalize()
    cycle = pd.Timedelta(days=train_days + validation_days)
    for i in range(n_splits):
        start = origin + i * cycle
        middle = start + pd.Timedelta(days=train_days)
        end = start + cycle
        train_idx = np.flatnonzero((data.timestamp >= start) & (data.timestamp < middle))
        validation_idx = np.flatnonzero((data.timestamp >= middle) & (data.timestamp < end))
        yield train_idx, validation_idx

In [ ]:
# сформируйте time_splits для расширяющегося окна: 5 блоков, test_size=300.
# для скользящего окна дополнительно задайте max_train_size=800.
# получите четыре блочных цикла через blocked_time_splits(time_pool).
# для каждой схемы вызовите analyze, проверьте время и пересечения групп.
# вычислите coverage и число событий, не использованных в validation.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, (name, splits) in zip(axes, time_splits.items()):
    for fold, (tr, va) in enumerate(splits, 1):
        ax.plot(time_pool.iloc[tr].timestamp, np.full(len(tr), fold), '.', color='steelblue', markersize=2,
                label='Train' if fold == 1 else None)
        ax.plot(time_pool.iloc[va].timestamp, np.full(len(va), fold), '.', color='darkorange', markersize=2,
                label='Validation' if fold == 1 else None)
    ax.set_title(name)
    ax.set_ylabel('Итерация')
    ax.set_yticks(range(1, len(splits) + 1))
    ax.legend(loc='upper left')
    ax.grid(alpha=0.25)
axes[-1].set_xlabel('Время регистрации события')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

**Ваш вывод:** Объясните три графика, неполное покрытие, размеры train и ограничения групп на границах времени.


## Задание 5. Выбор схемы для режима обучения и вывод SOC — 5 минут, 2 балла

1. Для обучения с учителем, без учителя и только по нормальным данным указать состав четырёх выборок и назначение меток
2. Для нормального режима выделить нормальные события хронологического train и calibration; validation и test сохранить смешанными
3. Составить записку SOC из 5–7 предложений: выбрать схему для будущих событий и для новых инцидентов, назвать её ограничения

**Проверяемый результат:** Таблица трёх режимов, нулевое число атак в normal_train и normal_calibration и обоснованная записка SOC.


In [ ]:
# выделите normal_train из chron_train и normal_calibration из time_calibration.
# оставьте chron_validation и test смешанными; проверьте отсутствие атак в нормальных частях.

Заполните таблицу: режим — train — validation — calibration — test — использование меток.

**Записка SOC (5–7 предложений):** …
